[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h05_flutter.ipynb)

# Why the Tacoma Narrows Bridge fell: one number

In 1940 a camera filmed a new bridge twisting itself apart in a steady wind. It wasn't resonance: the wind's swirls
came about once a second, and the bridge twisted once every five seconds. Instead, each twist set up the wind's next
push. Everything turns on one number: what each twist gets multiplied by. Below one, the twisting dies away. Above
one, it grows. Here you can set that number yourself, find the wind speed where it crosses one in a simple model,
check the bridge's own numbers, and try the challenge from the video.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

FT = 0.3048                                 # metres in a foot
TWIST_S = 5                                 # the bridge twisted about once every five seconds

print("Ready. Run the next cell.")

## 1 · One number, multiplied again and again

Each twist is the last one times a factor. Set the factor and the number of twists. Try 0.98 (the video's "two
percent smaller"), then 1.02 ("two percent bigger"), then 1.00 exactly.

In [ ]:
def twists(factor=0.98, n=100):
    sizes = [factor ** k for k in range(n + 1)]
    verdict = "dies away" if factor < 1 else "grows" if factor > 1 else "stays the same"
    print(f"after {n} twists: {sizes[-1]:.3f} times the first  ({verdict})")
    print(f"{n} twists at one every {TWIST_S} s take {n * TWIST_S / 60:.1f} minutes")
    t = [k * TWIST_S for k in range(n * 8 + 1)]
    swing = [factor ** (s / TWIST_S) * math.sin(2 * math.pi * s / TWIST_S) for s in (k / 8 for k in t)]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([k / 8 / 60 for k in t], swing, color=TEAL, lw=1.2, label="the twist, back and forth")
    ax.plot([k * TWIST_S / 60 for k in range(n + 1)], sizes, color=CORAL, lw=2.5, label=f"its size: {factor}ⁿ")
    ax.set_xlabel("minutes"); ax.set_ylabel("size of the twist (the first = 1)"); ax.legend(loc="upper left"); plt.show()

interact(twists, factor=FloatSlider(value=0.98, min=0.90, max=1.10, step=0.005, readout_format=".3f", description="factor"),
         n=IntSlider(value=100, min=10, max=300, description="twists"));

## 2 · A toy flutter model

This is a made-up model, not Gertie's real numbers. On every twist the steel and the air soak up a share of the
energy (the **damping**). The wind puts some back, more the faster it blows (the **feed**, a share for every 10 km/h).
So each twist is multiplied by

factor = 1 − damping + feed × (wind speed ÷ 10)

The **flutter speed** is the wind speed where the factor crosses one. Make the bridge lighter or flatter-sided (more
feed) or stiffer and better damped, and watch the flutter speed move.

In [ ]:
def flutter(damping=0.04, feed=0.007, wind=64):
    factor = 1 - damping + feed * wind / 10
    v_f = 10 * damping / feed
    print(f"wind {wind} km/h: each twist × {factor:.3f}  →  after 100 twists × {factor ** 100:.2f}")
    print(f"flutter speed in this model: {v_f:.0f} km/h  ({'below' if wind < v_f else 'above'} it)")
    vs = list(range(0, 161, 2))
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(vs, [1 - damping + feed * v / 10 for v in vs], color=TEAL, lw=2.5, label="the factor")
    ax.axhline(1, color=DIM, lw=1.5)
    ax.axvline(v_f, color=CORAL, lw=2, label=f"flutter speed ≈ {v_f:.0f} km/h")
    ax.scatter([wind], [factor], color=CHALK, zorder=3, label="your wind")
    ax.set_xlabel("wind speed (km/h)"); ax.set_ylabel("each twist multiplied by"); ax.legend(); plt.show()

interact(flutter, damping=FloatSlider(value=0.04, min=0.005, max=0.10, step=0.005, readout_format=".3f", description="damping"),
         feed=FloatSlider(value=0.007, min=0.001, max=0.02, step=0.001, readout_format=".3f", description="feed"),
         wind=IntSlider(value=64, min=0, max=160, step=2, description="wind km/h"));

## 3 · Resonance would need matching rhythms

Resonance means pushes in time with the bridge's own rhythm. That morning the wind's swirls came about once a second,
and the bridge twisted once every five seconds. Plot the two beats and see that they don't line up.

In [ ]:
def beats(seconds=20):
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.vlines([k for k in range(seconds + 1)], 0.6, 1.0, color=LILAC, lw=2, label="swirls: about 1 a second")
    ax.vlines([k for k in range(0, seconds + 1, TWIST_S)], 0.0, 0.4, color=TEAL, lw=4, label=f"twist: 1 every {TWIST_S} s")
    ax.set_yticks([]); ax.set_xlabel("seconds"); ax.legend(loc="upper right", ncol=2); ax.set_ylim(-0.1, 1.5); plt.show()
    print(f"in {seconds} s: {seconds} swirls, {seconds // TWIST_S} twists")

interact(beats, seconds=IntSlider(value=20, min=5, max=60, step=5, description="seconds"));

## 4 · The bridge's own numbers

The main span was 2,800 ft and the deck 39 ft wide. Along its sides were solid steel plates 8 ft deep; an earlier
design had open frames 25 ft deep. At its worst, one side of the road was 28 ft above the other.

In [ ]:
SPAN_FT, WIDTH_FT, RISE_FT = 2800, 39, 28
print(f"main span: {SPAN_FT:,} ft = {SPAN_FT * FT:.0f} m")
print(f"span ÷ width: {SPAN_FT / WIDTH_FT:.1f}  (about {SPAN_FT / WIDTH_FT:.0f} to 1)")
print(f"plate girders: 8 ft = {8 * FT:.2f} m   ·   the earlier frames: 25 ft = {25 * FT:.2f} m")
print(f"the tilt: one side {RISE_FT} ft above the other, across {WIDTH_FT} ft → asin({RISE_FT}/{WIDTH_FT}) = "
      f"{math.degrees(math.asin(RISE_FT / WIDTH_FT)):.1f}°")
print(f"the 1950 bridge's open trusses: 33 ft = {33 * FT:.1f} m deep")

## 5 · The challenge from the video

A twist starts at one degree, and every swing makes it five percent bigger. Roughly how many swings until it passes
forty-five degrees? Think first (or use the first cell with factor 1.05), then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("1° × 1.05 each swing: how many swings to pass 45°?   (press the button when you're ready)"); return
    n = next(k for k in range(1, 500) if 1.05 ** k > 45)
    print(f"after {n - 1} swings: {1.05 ** (n - 1):.2f}°   (just short)")
    print(f"after {n} swings: {1.05 ** n:.2f}°   (past 45°)")
    print(f"at one swing every {TWIST_S} s: {n * TWIST_S} s, about {n * TWIST_S / 60:.1f} minutes")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
Wikipedia: Tacoma Narrows Bridge (1940). K. Y. Billah and R. H. Scanlan, "Resonance, Tacoma Narrows bridge failure,
and undergraduate physics textbooks", *American Journal of Physics* 59 (1991) 118–124. APS News, "Collapse of the
Tacoma Narrows Bridge" (November 2016). WSDOT, Tacoma Narrows Bridge history. Carleton University case study. All
retrieved 2026-10-30. The flutter model in section 2 is a toy, made up to show the idea.